In [1]:

%pip install -q pandas numpy matplotlib seaborn plotly ipywidgets

import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, clear_output, HTML

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# =========================================================
# 1. LOAD AND PREPROCESS DATA
# =========================================================

FOLDER = r"C:\Users\mdnom\OneDrive\Desktop\umme hani projects\games"

athlete_path = os.path.join(FOLDER, "athlete_events.csv")
region_path = os.path.join(FOLDER, "noc_regions.csv")

if not os.path.exists(athlete_path):
    raise FileNotFoundError(f"Dataset not found: {athlete_path}")

if not os.path.exists(region_path):
    raise FileNotFoundError(
        "noc_regions.csv is missing. Download it from the "
        "same Kaggle Olympic dataset and put it in the games folder."
    )

df = pd.read_csv(athlete_path)
region_df = pd.read_csv(region_path)

# Match original CampusX preprocessing
df = df[df["Season"] == "Summer"].copy()
df = df.merge(region_df, on="NOC", how="left")
df = df.drop_duplicates().copy()

if "region" not in df.columns:
    raise ValueError("Country/region column was not created.")

df["region"] = df["region"].fillna(df["NOC"])

# Medal indicator columns, as in original preprocessing
medal_dummies = pd.get_dummies(df["Medal"])
for medal in ["Gold", "Silver", "Bronze"]:
    if medal not in medal_dummies.columns:
        medal_dummies[medal] = 0

df = pd.concat([df, medal_dummies], axis=1)

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Height"] = pd.to_numeric(df["Height"], errors="coerce")
df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")

print("Dataset loaded successfully!")
print("Summer Olympic records:", len(df))
print("Olympic years:", df["Year"].min(), "-", df["Year"].max())
print("Countries/regions:", df["region"].nunique())

# =========================================================
# 2. SHARED HELPER FUNCTIONS
# =========================================================

def medal_records(data):
    return data.dropna(subset=["Medal"]).drop_duplicates(
        subset=[
            "Team", "NOC", "Games", "Year", "City",
            "Sport", "Event", "Medal"
        ]
    ).copy()


def get_medal_tally(year, country):
    m = medal_records(df)

    if year != "Overall":
        m = m[m["Year"] == int(year)]

    if country != "Overall":
        m = m[m["region"] == country]

    if country != "Overall":
        result = m.groupby("Year")[["Gold", "Silver", "Bronze"]].sum()
        result = result.reset_index()
    else:
        result = m.groupby("region")[
            ["Gold", "Silver", "Bronze"]
        ].sum().reset_index()

    result["Total"] = result[
        ["Gold", "Silver", "Bronze"]
    ].sum(axis=1)

    if country == "Overall":
        result = result.sort_values(
            ["Gold", "Silver", "Bronze"], ascending=False
        )

    return result


def data_over_time(column):
    return (
        df.drop_duplicates(["Year", column])
        .groupby("Year")[column]
        .nunique()
        .reset_index(name="Count")
    )


def most_successful(sport="Overall"):
    data = medal_records(df)

    if sport != "Overall":
        data = data[data["Sport"] == sport]

    return (
        data.groupby("Name")
        .agg(
            Medal_Records=("Medal", "size"),
            Sport=("Sport", "first"),
            Country=("region", "first")
        )
        .sort_values("Medal_Records", ascending=False)
        .head(15)
        .reset_index()
    )


def country_medal_history(country):
    data = medal_records(df)
    data = data[data["region"] == country]

    return data.groupby("Year").size().reset_index(
        name="Medals"
    )


# =========================================================
# 3. NOTEBOOK DASHBOARD HEADER AND NAVIGATION
# =========================================================

display(HTML("""
<div style="
    background:linear-gradient(120deg,#082c57,#1266a5);
    color:white;padding:24px;border-radius:12px;
    margin-bottom:12px;">
    <h1 style="color:white;margin:0">🏅 Olympics Analysis</h1>
    <p style="margin:8px 0 0">
        Explore Olympic history, medals, countries and athletes
    </p>
</div>
"""))

menu = widgets.ToggleButtons(
    options=[
        "Medal Tally",
        "Overall Analysis",
        "Country-wise Analysis",
        "Athlete wise Analysis"
    ],
    description="Section:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
    button_style="info"
)

output = widgets.Output(
    layout=widgets.Layout(width="100%", padding="12px")
)

display(menu, output)

# =========================================================
# 4. MEDAL TALLY
# =========================================================

def show_medal_tally():
    years = ["Overall"] + sorted(
        df["Year"].dropna().astype(int).unique().tolist()
    )
    countries = ["Overall"] + sorted(
        df["region"].dropna().unique().tolist()
    )

    year_dropdown = widgets.Dropdown(
        options=years, value="Overall", description="Select Year:"
    )
    country_dropdown = widgets.Dropdown(
        options=countries, value="Overall",
        description="Select Country:"
    )
    result = widgets.Output()

    def update(*args):
        with result:
            clear_output(wait=True)
            table = get_medal_tally(
                year_dropdown.value, country_dropdown.value
            )

            if year_dropdown.value == "Overall" and country_dropdown.value == "Overall":
                title = "Overall Medal Tally"
            elif country_dropdown.value != "Overall":
                title = country_dropdown.value + " Medal Performance"
            else:
                title = "Medal Tally — " + str(year_dropdown.value)

            display(HTML(f"<h2>{title}</h2>"))
            if table.empty:
                print("No medal records found for this selection.")
            else:
                display(table.reset_index(drop=True).style.background_gradient(
                    subset=["Gold", "Silver", "Bronze", "Total"],
                    cmap="YlOrBr"
                ))

    year_dropdown.observe(update, names="value")
    country_dropdown.observe(update, names="value")

    display(widgets.HBox([year_dropdown, country_dropdown]))
    display(result)
    update()


# =========================================================
# 5. OVERALL ANALYSIS
# =========================================================

def show_overall():
    editions = df["Year"].nunique() - 1
    hosts = df["City"].nunique()
    sports = df["Sport"].nunique()
    events = df["Event"].nunique()
    athletes = df["Name"].nunique()
    nations = df["region"].nunique()

    metrics = [
        ("Editions", editions),
        ("Hosts", hosts),
        ("Sports", sports),
        ("Events", events),
        ("Nations", nations),
        ("Athletes", athletes)
    ]

    cards = "".join([
        f"""
        <div style="
          flex:1;min-width:115px;background:#edf5ff;
          padding:16px;border-radius:10px;text-align:center;
          border:1px solid #c8ddf5">
          <div style="font-size:14px;color:#334b67">{name}</div>
          <div style="font-size:26px;font-weight:bold;color:#0b4f86">
          {value:,}</div>
        </div>
        """ for name, value in metrics
    ])

    display(HTML(
        '<h2>Top Statistics</h2><div style="display:flex;'
        'flex-wrap:wrap;gap:10px">' + cards + '</div>'
    ))

    for column, title in [
        ("region", "Participating Nations Over the Years"),
        ("Event", "Events Over the Years"),
        ("Name", "Athletes Over the Years")
    ]:
        temp = data_over_time(column)
        temp.columns = ["Year", "Count"]

        fig = px.line(
            temp, x="Year", y="Count", markers=True,
            title=title
        )
        fig.show()

    display(HTML("<h2>Number of Events by Sport and Year</h2>"))

    event_data = df.drop_duplicates(["Year", "Sport", "Event"])
    pivot = event_data.pivot_table(
        index="Sport", columns="Year", values="Event",
        aggfunc="count"
    ).fillna(0)

    # Keep the 20 most active sports for a readable notebook chart
    pivot = pivot.loc[
        pivot.sum(axis=1).nlargest(20).index
    ]

    plt.figure(figsize=(16, 9))
    sns.heatmap(pivot, cmap="YlOrRd")
    plt.title("Olympic Events by Sport and Year (Top 20 Sports)")
    plt.xlabel("Year")
    plt.ylabel("Sport")
    plt.tight_layout()
    plt.show()

    display(HTML("<h2>Most Successful Athletes</h2>"))

    sport_dropdown = widgets.Dropdown(
        options=["Overall"] + sorted(df["Sport"].dropna().unique()),
        description="Select Sport:"
    )
    athlete_output = widgets.Output()

    def update_athletes(change=None):
        with athlete_output:
            clear_output(wait=True)
            display(most_successful(sport_dropdown.value))

    sport_dropdown.observe(update_athletes, names="value")
    display(sport_dropdown, athlete_output)
    update_athletes()


# =========================================================
# 6. COUNTRY-WISE ANALYSIS
# =========================================================

def show_country():
    countries = sorted(df["region"].dropna().unique().tolist())

    country_dropdown = widgets.Dropdown(
        options=countries, description="Select Country:",
        layout=widgets.Layout(width="500px")
    )
    country_output = widgets.Output()

    def update_country(change=None):
        country = country_dropdown.value

        with country_output:
            clear_output(wait=True)

            display(HTML(
                f"<h2>{country} — Medal Tally Over the Years</h2>"
            ))

            history = country_medal_history(country)

            if history.empty:
                print("No medal records found.")
            else:
                fig = px.line(
                    history, x="Year", y="Medals",
                    markers=True, title=f"{country} Medal History"
                )
                fig.show()

            display(HTML(
                f"<h2>{country} — Best Sports</h2>"
            ))

            country_data = medal_records(df)
            country_data = country_data[
                country_data["region"] == country
            ]

            heat = country_data.pivot_table(
                index="Sport", columns="Year", values="Medal",
                aggfunc="count", fill_value=0
            )

            if not heat.empty:
                heat = heat.loc[heat.sum(axis=1).nlargest(15).index]
                plt.figure(figsize=(14, 7))
                sns.heatmap(heat, cmap="YlGnBu")
                plt.title(f"{country}: Medal Records by Sport and Year")
                plt.tight_layout()
                plt.show()

            display(HTML(
                f"<h2>Top 10 Athletes of {country}</h2>"
            ))

            top = (
                country_data.groupby("Name")
                .agg(
                    Medal_Records=("Medal", "size"),
                    Sport=("Sport", "first")
                )
                .sort_values("Medal_Records", ascending=False)
                .head(10)
                .reset_index()
            )
            display(top)

    country_dropdown.observe(update_country, names="value")
    display(country_dropdown, country_output)
    update_country()


# =========================================================
# 7. ATHLETE-WISE ANALYSIS
# =========================================================

def show_athletes():
    athlete_df = df.drop_duplicates(
        subset=["Name", "region"]
    ).copy()

    display(HTML("<h2>Distribution of Age</h2>"))

    age_data = athlete_df.dropna(subset=["Age"]).copy()
    age_data["Medal Group"] = age_data["Medal"].fillna("No Medal")

    fig = px.histogram(
        age_data, x="Age", color="Medal Group",
        barmode="overlay", opacity=0.65,
        title="Age Distribution by Medal Category"
    )
    fig.show()

    display(HTML("<h2>Height vs Weight</h2>"))

    sport_dropdown = widgets.Dropdown(
        options=["Overall"] + sorted(
            df["Sport"].dropna().unique().tolist()
        ),
        description="Select Sport:"
    )
    scatter_output = widgets.Output()

    def update_scatter(change=None):
        with scatter_output:
            clear_output(wait=True)

            data = athlete_df.dropna(
                subset=["Height", "Weight"]
            ).copy()

            if sport_dropdown.value != "Overall":
                data = data[
                    data["Sport"] == sport_dropdown.value
                ]

            data["Medal Group"] = data["Medal"].fillna("No Medal")
            data = data.sample(
                n=min(3000, len(data)), random_state=42
            )

            if not data.empty:
                fig = px.scatter(
                    data, x="Weight", y="Height",
                    color="Medal Group", symbol="Sex",
                    hover_data=["Sport"],
                    title="Athlete Height vs Weight"
                )
                fig.show()
            else:
                print("No height/weight records available.")

    sport_dropdown.observe(update_scatter, names="value")
    display(sport_dropdown, scatter_output)
    update_scatter()

    display(HTML(
        "<h2>Men vs Women Participation Over the Years</h2>"
    ))

    unique_athletes = df.drop_duplicates(
        subset=["Name", "region", "Year", "Sport"]
    )

    participation = unique_athletes.pivot_table(
        index="Year", columns="Sex", values="Name",
        aggfunc="nunique", fill_value=0
    ).reset_index()

    participation = participation.rename(
        columns={"M": "Male", "F": "Female"}
    )

    fig = px.line(
        participation, x="Year",
        y=[c for c in ["Male", "Female"] if c in participation.columns],
        markers=True,
        title="Male and Female Participation"
    )
    fig.show()


# =========================================================
# 8. NAVIGATION: SHOW THE SELECTED SECTION
# =========================================================

def render_dashboard(change=None):
    with output:
        clear_output(wait=True)

        if menu.value == "Medal Tally":
            show_medal_tally()

        elif menu.value == "Overall Analysis":
            show_overall()

        elif menu.value == "Country-wise Analysis":
            show_country()

        elif menu.value == "Athlete wise Analysis":
            show_athletes()


menu.observe(render_dashboard, names="value")

# Show the first dashboard section
render_dashboard()



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Dataset loaded successfully!
Summer Olympic records: 221167
Olympic years: 1896 - 2016
Countries/regions: 209


ToggleButtons(button_style='info', description='Section:', layout=Layout(width='100%'), options=('Medal Tally'…

Output(layout=Layout(padding='12px', width='100%'))

In [2]:
%pip install --user pandas numpy matplotlib seaborn plotly ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
